# Optimización por ballenas (WOA)

Adaptación del cuaderno CLONALG: minimización continua, visualización y exportación de video. La figura muestra dos etapas por iteración: población anterior y movimientos.


In [18]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter

plt.rcParams['animation.embed_limit'] = 300


def evaluar(poblacion, funcion):
    """Evalúa una función objetivo de minimización para cada solución."""
    valores = np.array([funcion(x) for x in poblacion], dtype=float)
   
    return valores


def woa(
    funcion,
    limites,
    tamano=30,
    iteraciones=100,
    semilla=42,
    b=1.0
):
    """Algoritmo de Optimización de Ballenas (WOA) para minimización."""

    limites = np.asarray(limites, dtype=float)

    rng = np.random.default_rng(semilla)
    inferior, superior = limites[:, 0], limites[:, 1]
    dimension = len(limites)

    # 1. INICIALIZACIÓN
    # Cada ballena representa una solución aleatoria dentro del espacio de búsqueda.
    poblacion = rng.uniform(inferior, superior, (tamano, dimension))

    # Evaluamos la población y seleccionamos la mejor ballena inicial.
    errores = evaluar(poblacion, funcion)

    indice = np.argmin(errores)
    mejor_global = poblacion[indice].copy()
    error_global = float(errores[indice])

    historial = [error_global]
    instantaneas = [poblacion.copy()]
    etapas = []

    # 2. ITERACIONES DEL ALGORITMO
    for paso in range(iteraciones):

        # Conservamos las posiciones anteriores para la animación.
        anterior = poblacion.copy()
        mejor_anterior = mejor_global.copy()

        # Aquí se almacenarán las nuevas posiciones de las ballenas.
        nueva = np.empty_like(anterior)
        movimientos = []

        # El parámetro 'a' disminuye linealmente de 2 hacia 0.
        # Esto reduce progresivamente la exploración y favorece la explotación.
        a = 2 * (1 - paso / iteraciones)

        # Cada ballena actualiza su posición.
        for i, ballena in enumerate(anterior):

            # Coeficientes del WOA:
            # A controla la exploración o explotación.
            # C introduce variabilidad en el cálculo de la distancia.
            r = rng.random()
            A = 2 * a * r - a
            C = 2 * r

            # p determina el tipo de movimiento:
            # p < 0.5: exploración o rodeo.
            # p >= 0.5: ataque en espiral.
            p = rng.random()

            if p < 0.5:

                if abs(A) >= 1:
                    # 3. EXPLORACIÓN: BÚSQUEDA DE PRESAS
                    # Se selecciona una ballena aleatoria como referencia,
                    # permitiendo explorar diferentes regiones del espacio.
                    referencia = anterior[rng.integers(tamano)]
                    movimientos.append('exploracion')

                else:
                    # 4. EXPLOTACIÓN: RODEAR A LA PRESA
                    # La ballena se mueve tomando como referencia
                    # la mejor solución encontrada hasta el momento.
                    referencia = mejor_anterior
                    movimientos.append('rodeo')

                # Distancia respecto a la ballena de referencia.
                distancia = np.abs(C * referencia - ballena)

                # Nueva posición mediante exploración o rodeo.
                nueva[i] = referencia - A * distancia

            else:
                # 5. EXPLOTACIÓN: ATAQUE EN ESPIRAL
                # La ballena realiza un movimiento matemático en espiral
                # alrededor de la mejor solución encontrada.

                # l es un valor aleatorio entre -1 y 1.
                # b determina la forma de la espiral.
                l = rng.uniform(-1, 1)

                # Distancia respecto a la mejor solución.
                distancia = np.abs(mejor_anterior - ballena)

                # Ecuación del movimiento en espiral.
                nueva[i] = (
                    distancia
                    * np.exp(b * l)
                    * np.cos(2 * np.pi * l)
                    + mejor_anterior
                )

                movimientos.append('espiral')

        # 6. ACTUALIZACIÓN DE LA POBLACIÓN
        # Evitamos que las ballenas salgan del espacio de búsqueda.
        poblacion = np.clip(nueva, inferior, superior)

        # Evaluamos las nuevas posiciones para encontrar la mejor solución.
        errores = evaluar(poblacion, funcion)
        indice = np.argmin(errores)

        # Actualizamos la mejor solución global solamente si encontramos
        # una ballena con un resultado menor al anterior.
        if errores[indice] < error_global:
            mejor_global = poblacion[indice].copy()
            error_global = float(errores[indice])

        # Guardamos el progreso del algoritmo.
        historial.append(error_global)
        instantaneas.append(poblacion.copy())

        # Guardamos los datos necesarios para la animación con Matplotlib.
        etapas.append({
            'anterior': anterior,
            'poblacion': poblacion.copy(),
            'movimientos': np.array(movimientos),
            'mejor_anterior': mejor_anterior,
            'mejor': mejor_global.copy(),
            'error': error_global,
            'a': a
        })

    # 7. RESULTADO FINAL
    # Devolvemos la mejor solución y la información para visualizar el proceso.
    return {
        'posicion': mejor_global,
        'error': error_global,
        'historial': np.array(historial),
        'instantaneas': instantaneas,
        'etapas': etapas,
        'limites': limites,
        'funcion': funcion
    }



In [19]:
def animar_woa(resultado, funcion=None, intervalo=250, guardar=None,
               cada=1, mostrar=True, minimo_teorico=None):
    """Anima WOA y su convergencia; guardar admite rutas .gif o .mp4.

    Para más de dos dimensiones se muestra un corte del paisaje en la mejor
    solución final y las ballenas se proyectan sobre las dos primeras variables.
    """
    funcion = funcion if funcion is not None else resultado.get('funcion')
    if funcion is None:
        raise ValueError('Debes proporcionar funcion.')
    limites = np.asarray(resultado['limites'])
    if len(limites) < 2:
        raise ValueError('La animación requiere al menos dos dimensiones.')
    if not isinstance(cada, int) or cada < 1 or intervalo <= 0:
        raise ValueError('cada debe ser entero >= 1 e intervalo debe ser > 0.')
    if not resultado['etapas']:
        raise ValueError('No hay iteraciones para animar.')
    x = np.linspace(*limites[0], 160)
    y = np.linspace(*limites[1], 160)
    X, Y = np.meshgrid(x, y)
    coordenadas = np.tile(resultado['posicion'], (X.size, 1))
    coordenadas[:, 0], coordenadas[:, 1] = X.ravel(), Y.ravel()
    Z = evaluar(coordenadas, funcion).reshape(X.shape)
    zmin, zmax = float(Z.min()), float(Z.max())
    if zmin >= 0 and zmax > 0:
        vmin = max(min(0.01, zmax / 1000), 1e-15)
        dibujo = np.maximum(Z, vmin)
        norm = LogNorm(vmin=vmin, vmax=max(zmax, vmin * 10))
    else:
        dibujo = Z
        norm = Normalize(vmin=zmin, vmax=zmax if zmax > zmin else zmin + 1)
    fig, (ax, ax_error) = plt.subplots(1, 2, figsize=(16, 7),
                                      gridspec_kw={'width_ratios': [1.4, 1]})
    fig.subplots_adjust(wspace=0.34, bottom=0.18)
    imagen = ax.pcolormesh(X, Y, dibujo, shading='auto', cmap='viridis', norm=norm)
    niveles = [v for v in [0.01, 0.1, 1, 10, 100, 1000, 10000] if zmin < v < zmax]
    if niveles:
        ax.contour(X, Y, Z, levels=niveles, colors='white', alpha=0.4, linewidths=0.65)
    fig.colorbar(imagen, ax=ax, label='f(x) (corte 2D)', fraction=0.046, pad=0.04)
    ax.set(xlim=limites[0], ylim=limites[1], xlabel='x[0]', ylabel='x[1]')
    if minimo_teorico is not None:
        minimo = np.asarray(minimo_teorico, dtype=float)
        if minimo.ndim != 1 or len(minimo) < 2:
            raise ValueError('minimo_teorico debe tener al menos dos coordenadas.')
        ax.plot(minimo[0], minimo[1], 'w+', ms=10, mew=2, label='Mínimo teórico')
    if len(limites) > 2:
        ax.text(0.5, -0.14, 'Corte 2D: variables restantes fijadas en la mejor solución final.\n'
                'Las ballenas son proyecciones.', transform=ax.transAxes, ha='center', fontsize=8)
    historial = np.asarray(resultado['historial'])
    log = np.all(historial >= 0)
    hist = np.maximum(historial, 1e-12) if log else historial
    ax_error.plot(np.arange(len(hist)), hist, color='0.7', lw=1, label='Historial completo')
    progreso, = ax_error.plot([], [], color='tab:red', lw=2, label='Hasta la iteración actual')
    indicador, = ax_error.plot([], [], 'o', color='black')
    ax_error.set(xlabel='Iteración', ylabel='Mejor valor global', title='Convergencia',
                 xlim=(0, max(1, len(historial) - 1)))
    ax_error.set_yscale('log' if log else 'symlog')
    ax_error.legend(fontsize=8)
    pasos = list(range(0, len(resultado['etapas']), cada))
    if pasos[-1] != len(resultado['etapas']) - 1:
        pasos.append(len(resultado['etapas']) - 1)
    fotogramas = [(i, fase) for i in pasos for fase in range(2)]
    colores = {'exploracion': 'deepskyblue', 'rodeo': 'lime', 'espiral': 'magenta'}
    nombres = {'exploracion': 'Exploración', 'rodeo': 'Rodeo', 'espiral': 'Espiral'}
    artistas = []
    def actualizar(numero):
        for artista in artistas:
            artista.remove()
        artistas.clear()
        i, fase = fotogramas[numero]
        d = resultado['etapas'][i]
        if fase == 0:
            artistas.append(ax.scatter(d['anterior'][:, 0], d['anterior'][:, 1],
                                       c='black', s=24, alpha=0.7, label='Ballenas'))
            mejor = d['mejor_anterior']
            titulo = 'Población anterior'
            fin_historial = i
        else:
            for tipo, color in colores.items():
                indices = np.flatnonzero(d['movimientos'] == tipo)
                if len(indices):
                    for j in indices:
                        linea, = ax.plot([d['anterior'][j, 0], d['poblacion'][j, 0]],
                                         [d['anterior'][j, 1], d['poblacion'][j, 1]],
                                         color=color, alpha=0.5, lw=0.8)
                        artistas.append(linea)
                    pts = d['poblacion'][indices]
                    artistas.append(ax.scatter(pts[:, 0], pts[:, 1], c=color, s=35,
                                               edgecolors='black', linewidths=0.4, label=nombres[tipo]))
            mejor = d['mejor']
            titulo = 'Movimientos y nueva población'
            fin_historial = i + 1
        artistas.append(ax.scatter(mejor[0], mejor[1], c='gold', s=200, marker='*',
                                   edgecolors='black', label='Mejor global', zorder=5))
        ax.set_title(f'WOA | Iteración {i + 1} | {titulo}')
        ax.legend(loc='upper left', fontsize=8)
        progreso.set_data(np.arange(fin_historial + 1), hist[:fin_historial + 1])
        indicador.set_data([fin_historial], [hist[fin_historial]])
        ax_error.set_title(f'Mejor valor: {historial[fin_historial]:.3g}')
        return artistas
    animacion = FuncAnimation(fig, actualizar, frames=len(fotogramas),
                              interval=intervalo, repeat=True, blit=False)
    if guardar:
        ruta = str(guardar)
        fps = max(1, round(1000 / intervalo))
        if ruta.lower().endswith('.gif'):
            writer = PillowWriter(fps=fps)
        elif ruta.lower().endswith('.mp4'):
            try:
                import imageio_ffmpeg
                plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
            except ImportError as exc:
                raise ImportError('Para MP4 instala imageio-ffmpeg: pip install imageio-ffmpeg') from exc
            writer = FFMpegWriter(fps=fps)
        else:
            raise ValueError('Usa una ruta terminada en .gif o .mp4.')
        animacion.save(ruta, writer=writer, dpi=120)
        print('Animación guardada en:', ruta)
    if mostrar:
        plt.show()
    else:
        plt.close(fig)
    return animacion




In [20]:
from IPython.display import HTML, display

# 1. RASTRIGIN
# Espacio de búsqueda: [-5.12, 5.12]
# Dimensiones: N
# Mínimo global: f(0, 0, ..., 0) = 0
def rastrigin(x):
    return 10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))


# 2. BOOTH
# Espacio de búsqueda: [-10, 10]
# Dimensiones: 2
# Mínimo global: f(1, 3) = 0
def booth(x):
    return (x[0] + 2*x[1] - 7)**2 + (2*x[0] + x[1] - 5)**2


# 3. SPHERE
# Espacio de búsqueda: [-5.12, 5.12]
# Dimensiones: N
# Mínimo global: f(0, 0, ..., 0) = 0
def sphere(x):
    return np.sum(x**2)


# 4. BEALE
# Espacio de búsqueda: [-4.5, 4.5]
# Dimensiones: 2
# Mínimo global: f(3, 0.5) = 0
def beale(x):
    return (
        (1.5 - x[0] + x[0]*x[1])**2
        + (2.25 - x[0] + x[0]*x[1]**2)**2
        + (2.625 - x[0] + x[0]*x[1]**3)**2
    )


# 5. ACKLEY
# Espacio de búsqueda: [-32.768, 32.768]
# Dimensiones: 2 (esta implementación)
# Mínimo global: f(0, 0) = 0
def ackley(x):
    return (
        -20 * np.exp(-0.2 * np.sqrt(0.5 * (x[0]**2 + x[1]**2)))
        - np.exp(0.5 * (np.cos(2*np.pi*x[0]) + np.cos(2*np.pi*x[1])))
        + np.e + 20
    )


dimensiones = 2
limites = [[-10, 10]] * dimensiones
result = woa(funcion=booth, limites=limites, iteraciones=100, tamano=20, semilla=None)
print('Mejor posición:', result['posicion'])
print('Mejor error:', result['error'])
# cada=2 muestra una de cada dos iteraciones e incluye siempre la última.
animacion = animar_woa(result, cada=1, intervalo=250, mostrar=False,
                       minimo_teorico=np.zeros(dimensiones))
display(HTML(animacion.to_jshtml()))


Mejor posición: [1.89863881 2.11192356]
Mejor error: 1.5966777306908084


## Exportar el video

Ejecuta la instalación si aún no tienes la dependencia. El MP4 se guarda en la carpeta de trabajo del notebook.


In [21]:
%pip install imageio-ffmpeg


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import imageio_ffmpeg
import matplotlib as mpl

mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
animacion.save("woa.mp4", writer="ffmpeg", fps=4, dpi=120)
print("Video guardado en: woa.mp4")


Video guardado en: woa.mp4
